# Reproducing DocLens (Zhu et al., ACL 2026) on Kaggle

This notebook re-runs the experiments of *DocLens: A Tool-Augmented Multi-Agent Framework for Long Visual Document
Understanding* with the **official DocLens code** (https://github.com/dwzhu-pku/DocLens), using the paper's settings
(Appendix D.1), and prints every result next to the number in the paper.

| Tier | Paper experiments | Runs |
|---|---|---|
| 1 | **Table 1, Gemini-2.5-Pro**: Vanilla, +OCR, **DocLens (67.6)** | 3 |
| 2 | Table 1, Gemini-2.5-Flash: Vanilla, +OCR, DocLens | 3 |
| 3 | Ablations with Pro: Table 2 (w/o Lens, w/o Reasoning), Table 3 (navigator backbones, oracle pages), Table 4 (w/o sampling, w/o OCR), Table 6 (K=2) | 8 |
| 4 | Ablations with Flash / Flash-Lite: Tables 2, 4, 11 | 6 |

The page-retrieval numbers (Tables 3–4) and the per-domain breakdown (Table 10) are computed from the same runs.

**Settings taken from the paper** (Appendix D.1 and Table 3):
* Te = Ta = **8** samples, temperature **0.7**, MinerU parsing, 200-DPI screenshots.
* The Page Navigator uses the **same backbone** as the Reasoning Module.
* The Adjudicator prompt includes the paper's "Rule of Common Sense".

The **released repository differs from the paper in all three**: `main.py` hard-codes Gemini-2.5-Flash-Lite as navigator, the
end-to-end script uses K=2, and the prompt lacks that rule. The repository's defaults match the paper's cheaper "Hybrid" variant
(Table 6, 64.0), not the headline 67.6.

**Not re-runnable here** (cite the paper):
* the Claude-4-Sonnet rows: the released Claude code path crashes;
* SimpleDoc, MACT, M3DocRAG, MDocAgent, GPT-4o and o4-mini;
* the human-expert score;
* Figure 4: its block-level prompts are not released;
* latency (Tables 5–6).

Qwen3-VL-8B (Table 9) needs a separate API account.

### What "same results" requires
1. **The same models.** Gemini 2.5 Pro / Flash / Flash-Lite are no longer offered to new Gemini API keys. **Vertex AI** (Google Cloud) is
   the most likely route; section 3 tests every model. If one is missing, map it to a substitute in `MODEL_MAP`. The report then counts as
   a *re-run*, not a reproduction.
2. **Tolerance.** Answers are sampled at temperature 0.7, so expect differences of about ±1–2 points, not identical digits.

### Before running
* *Settings → Accelerator*: **GPU T4 x2** · *Internet*: **On**.
* *Add-ons → Secrets*: **`GCP_SA_KEY`**. This is the full JSON of a Google Cloud service-account key with the *Vertex AI User* role. With `ACCESS = "api_key"`, use `GEMINI_API_KEY` instead.
* Cost: the paper measures about **55k tokens per question** for DocLens with K=8 (Table 5), and MMLongBench-Doc has 1,082 questions.
  Section 7 measures this on 20 questions first; check your billing page before section 8.

In [ ]:
# ======================= SETTINGS =======================
WORK = "/kaggle/working"
CDC_REPO = "https://github.com/Mariam1543/aqqal123.git"
CDC_BRANCH = "claude/wizardly-rubin-kdthhy"

DATASET = "MMLongBenchDoc"      # or "FinRAGBench-V" (run the notebook once per benchmark)
ACCESS = "vertex"               # "vertex" (Google Cloud, original DocLens setup) or "api_key"
GCP_LOCATION = "us-east1"       # DocLens' default Vertex location

TIERS = [1]                     # start with [1]; add 2, 3, 4 when tier 1 matches the paper
MODEL_MAP = {}                  # only if a paper model is unavailable, e.g. {"gemini-2.5-pro": "gemini-3.8-pro"}
EVAL_MODEL = "gemini-2.5-flash" # answer extraction / judge used by DocLens' evaluation code
PAPER_PROMPTS = True            # add the paper's "Rule of Common Sense" to the Adjudicator (see intro)

PILOT_QUESTIONS = 20
RPM = 60                        # requests/minute cap; raise if your quota allows
MAX_CONCURRENT = 8

## 1. Get the code

In [ ]:
import os, sys, subprocess, shutil, json, glob
os.chdir(WORK)
if not os.path.exists("DocLens"):
    subprocess.run(["git", "clone", "https://github.com/dwzhu-pku/DocLens.git"], check=True)
if not os.path.exists("doclens_cdc/reproduction/experiments.py"):
    shutil.rmtree("doclens_cdc", ignore_errors=True)
    shutil.rmtree("_cdc_repo", ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", "-b", CDC_BRANCH, CDC_REPO, "_cdc_repo"], check=True)
    shutil.copytree("_cdc_repo/doclens_cdc", "doclens_cdc")
    shutil.rmtree("_cdc_repo")

DOCLENS, CDC = f"{WORK}/DocLens", f"{WORK}/doclens_cdc"
ROOT = f"{WORK}/full_{DATASET}"           # rendered + parsed benchmark (reused for the novelty experiments)
RESULTS = f"{WORK}/results_repro/{DATASET}"
os.makedirs(RESULTS, exist_ok=True)
print(subprocess.run(["git", "-C", DOCLENS, "log", "-1", "--format=DocLens commit %h (%cd)"],
                     capture_output=True, text=True).stdout)

sys.path.insert(0, CDC)
from reproduction.experiments import EXPERIMENTS, run_args, substitute
SELECTED = {k: substitute(e, MODEL_MAP) for k, e in EXPERIMENTS.items() if e["tier"] in TIERS}
for k, e in SELECTED.items():
    print(f"{k:24s} {e['note']:50s} model={e['model']} navigator={e['navigator']} K={e['k1']}/{e['k2']}")

## 2. Install
MinerU **2.x** is required (MinerU 3/4 replaced the command-line interface DocLens uses).

In [ ]:
!pip install -q "mineru[core]==2.7.6" 2>&1 | tail -n 2
!pip install -q -r {CDC}/requirements-kaggle.txt 2>&1 | tail -n 2
!mineru --version

## 3. Credentials and model availability
This decides whether the results count as a **reproduction** (the paper's models) or a **re-run with substitute models**.

In [ ]:
from kaggle_secrets import UserSecretsClient
from google import genai
secrets = UserSecretsClient()

if ACCESS == "vertex":
    key_json = secrets.get_secret("GCP_SA_KEY")
    key_path = "/tmp/gcp_key.json"          # outside /kaggle/working, so it is never saved with the outputs
    with open(key_path, "w") as f:
        f.write(key_json)
    os.chmod(key_path, 0o600)
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = key_path
    os.environ["GOOGLE_CLOUD_PROJECT"] = json.loads(key_json)["project_id"]
    os.environ["GOOGLE_CLOUD_LOCATION"] = GCP_LOCATION
    client = genai.Client(vertexai=True, project=os.environ["GOOGLE_CLOUD_PROJECT"], location=GCP_LOCATION)
    ACCESS_ARGS = "--use_vertex"
else:
    os.environ["GEMINI_API_KEY"] = secrets.get_secret("GEMINI_API_KEY")
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    ACCESS_ARGS = ""

needed = sorted({m for e in SELECTED.values() for m in (e["model"], e["navigator"])} | {EVAL_MODEL})
available = {}
for name in needed:
    try:
        client.models.generate_content(model=name, contents="Reply with OK.")
        available[name] = True
        print(f"  OK       {name}")
    except Exception as e:
        available[name] = False
        print(f"  FAILED   {name}: {str(e)[:200]}")

SAME_MODELS = all(available.values()) and not MODEL_MAP and EVAL_MODEL == "gemini-2.5-flash"
print()
if not all(available.values()):
    print("STOP: some models are not callable. Fix access, or map them to substitutes in MODEL_MAP and re-run sections 1 and 3.")
elif SAME_MODELS:
    print("All paper models are available: these runs are a REPRODUCTION.")
else:
    print("Substitute models in use: report these runs as a RE-RUN with the substitute models, not as a reproduction.")

## 4. Download the benchmark

In [ ]:
!python {CDC}/scripts/download_data.py --dataset {DATASET} --out {DOCLENS}/data
SAMPLES = f"{DOCLENS}/data/{DATASET}/samples.json"
DOCUMENTS = f"{DOCLENS}/data/{DATASET}/documents"
assert os.path.exists(SAMPLES), glob.glob(f"{DOCLENS}/data/{DATASET}/**/*.json", recursive=True)[:10]
N_EXPECTED = len(json.load(open(SAMPLES)))
print(N_EXPECTED, "questions (paper: 1,082 for MMLongBench-Doc, 1,394 for FinRAGBench-V);",
      len(glob.glob(f"{DOCUMENTS}/*.pdf")), "PDFs (paper: 135 / 301)")
print("Fields of one question:", sorted(json.load(open(SAMPLES))[0].keys()))

## 5. Render every page at 200 DPI (DocLens' setting)

In [ ]:
!python {CDC}/scripts/prepare_subset.py --all --samples {SAMPLES} --documents {DOCUMENTS} --out_root {ROOT} \
    --dataset_name {DATASET} --workers 4
QUESTIONS = f"{ROOT}/samples_subset.json"
!du -sh {ROOT}

## 6. Parse every page with MinerU (GPU, a few hours)
This is the OCR and layout input of the Page Navigator and Element Localizer. It **resumes** where it stopped: just run the cell again.

**Save this work.** When the cell finishes, click *Save Version → Quick Save*. In later sessions, add this notebook's output as an input
(*Add Input → Your Work*) and copy `full_<dataset>/` back to `/kaggle/working`, so you never have to parse again.

In [ ]:
!python {CDC}/scripts/parse_mineru.py --root {ROOT}/clean --dataset_name {DATASET} --batch_size 400
n_pages = len(glob.glob(f"{ROOT}/clean/data/{DATASET}/documents/*/*.jpeg"))
n_md = len(glob.glob(f"{ROOT}/clean/data/{DATASET}/documents/*/MinerU_Page*/*.md"))
print(f"{n_md}/{n_pages} pages parsed")
!du -sh {ROOT}

## 7. Pilot: 20 questions of the first selected experiment
Checks that everything works and measures tokens per question, so you can estimate the cost before the full runs.

In [ ]:
common = (f"python {CDC}/run_cdc.py --doclens_root {DOCLENS} --data_root {ROOT}/clean --variant clean "
          f"--dataset_name {DATASET} --samples {QUESTIONS} --eval_model {EVAL_MODEL} "
          f"--rpm {RPM} --max_concurrent {MAX_CONCURRENT} {ACCESS_ARGS} {'--paper_prompts' if PAPER_PROMPTS else ''}")
pilot_key = "pro_doclens" if "pro_doclens" in SELECTED else next(iter(SELECTED))
pilot_args = run_args(SELECTED[pilot_key])
!{common} {pilot_args} --limit {PILOT_QUESTIONS} --output {RESULTS}/pilot_{pilot_key}.jsonl

In [ ]:
rows = [json.loads(l) for l in open(f"{RESULTS}/pilot_{pilot_key}.jsonl")]
ok = [r for r in rows if "final_score" in r]
print(f"{pilot_key}: {len(ok)}/{len(rows)} scored, accuracy {sum(r['final_score'] for r in ok) / max(1, len(ok)):.3f}")
tok = sum(r.get("final_tokens", 0) for r in ok) / max(1, len(ok))
print(f"~{tok:,.0f} tokens per question (paper, Table 5: ~55,000 for K=8) "
      f"-> ~{tok * N_EXPECTED / 1e6:,.0f}M tokens for this experiment on the full benchmark")
for r in rows:
    if "cdc_error" in r:
        print("ERROR:", r["cdc_error"][:500]); break

## 8. Run the selected experiments on the full benchmark
Each experiment writes `results_repro/<dataset>/<experiment>.jsonl`. Re-running the cell **resumes**: finished questions are skipped,
and failed ones are retried.

In [ ]:
for key, exp in SELECTED.items():
    print("=" * 30, key, "-", exp["note"])
    args = run_args(exp)
    !{common} {args} --output {RESULTS}/{key}.jsonl

## 9. Compare with the paper

In [ ]:
!python {CDC}/scripts/report_reproduction.py --results_dir {RESULTS} --dataset_name {DATASET} \
    --expected_questions {N_EXPECTED} --out {RESULTS}/reproduction_report.md
print("\nReproduction (paper models):", SAME_MODELS, "| model substitutions:", MODEL_MAP or "none",
      "| paper prompts:", PAPER_PROMPTS)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{RESULTS}/reproduction_report.md").read()))

## 10. Next: your novelty on the same data
`full_<dataset>/clean` is the starting point for the confidence-gated abstention experiments. Create degraded copies, parse them with
MinerU, and run `--mode analyze` with the **same models, K and prompts** as the `pro_doclens` row:

```
!python {CDC}/scripts/make_variants.py --out_root {ROOT} --dataset_name {DATASET} --variants blur_s2 noise_s2 jpeg_s3 lowres_s2 mixed_s2
!python {CDC}/scripts/parse_mineru.py --root {ROOT}/blur_s2 --dataset_name {DATASET}
!{common} {run_args(SELECTED['pro_doclens']).replace('--mode doclens', '--mode analyze')} --data_root {ROOT}/blur_s2 --variant blur_s2 --output ...
```
Running `--mode analyze` on `clean` as well shows whether the extension changes anything on clean documents.